# 04. 상담 예제 3 — "해외 결제 승인 알림을 확인하고 싶어요" (risk 검증 sub-flow)

세 번째 상담 예제입니다. tool은 성공하지만 결과에 **fraud/AML 모니터링 flag**가 붙어 있어, D7(리스크 점검)이 이를 감지하면 **KYC 재검증 tool 실행 -> D8(통과 여부 판정)** 을 거쳐 종결하는 조건부 sub-flow를 돕니다.

후반부에는 **guardrail(D1) 민감도와 calibrated confidence**를 직접 체험합니다 — 같은 상황을 고객이 실제로 말할 법한 피해 신고 문구로 바꿔 보내고, 두 backend의 판정과 confidence가 어떻게 달라지는지 비교합니다.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로 (repo 내 상대 위치)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.SCENARIOS[3]
print("시나리오:", sc["title"])
print("고객 질문:", sc["question"])
print("script:", sc["script"])   # risk_flag: True, kyc_result: verified

## 전체 workflow에서 이 시나리오의 위치

이번 시나리오가 지나는 경로(빨간 테두리): tool 성공 결과에 모니터링 flag가 있어 **risk_check -> kyc_verify(D8)** sub-flow를 지나 종결합니다. 금융권의 "고위험 거래는 신원 재검증 후 종결" 절차가 그래프에 들어온 형태입니다.

배정 상담팀(빨간 표시): **card_security(분실/도난/보안)** — 이상 결제 확인·차단 문맥입니다. 실행에서 모델이 cards를 고르는 경우도 있으니 STEP 출력의 D2 판정과 비교해 보세요.
> 참고: 이 다이어그램은 실행되는 LangGraph에서 자동 생성된 것이 아니라, `ws_common.py`의 한글 캡션 사전(`KO_NODES`/`KO_EDGES`)을 노드 이름 기준으로 그린 **시각화 전용 그림**입니다 (상세 설명: `[KR]_01_build_agent.ipynb` 참조).


In [ ]:
W.draw_graph(highlight=W.SCENARIO_PATHS[3], show_teams=True, width=760)

## ① Haiku 4.5 판단으로 실행

STEP 출력에서 risk_check가 `needs_verification`을 판정한 뒤 **kyc_verify 노드가 추가로 실행**되는 것을 확인하세요.

In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## ② strands-decider(E4B) 판단으로 실행

In [ ]:
res_strands = W.run_verbose(graphs["StrandsDecider"], sc["question"], sc["script"], "StrandsDecider")

## ③ 수행시간 비교

In [ ]:
W.compare(res_haiku, res_strands)

## ④ (교보재) guardrail 민감도 — 피해 신고 문구를 양쪽 backend로 체험

위 시나리오의 질문은 일부러 중립적으로 썼습니다. 이번에는 같은 상황을 고객이 실제로 말할 법한 **피해 신고 문구**("제가 쓰지 않은 결제예요... 막아주세요")로 바꿔 양쪽 arm에 그대로 흘려 봅니다.

안전 판단(D1)은 소형 decision model에게 가장 까다로운 결정입니다 — 피해 *신고*와 compliance *위반*은 표면 어휘가 비슷하기 때문입니다. 범용 소형 모델을 zero-shot으로 쓰면 이런 문구를 과차단(위반으로 오판)하는 사례가 흔합니다. strands-decider가 이 문구를 어떻게 판정하는지, 그리고 **confidence가 중립 문구 대비 어떻게 달라지는지** 관찰하세요.

In [ ]:
victim_q = ("방금 제가 쓰지 않은 해외 결제 승인 알림이 왔어요. LUXE GADGETS라는 곳에서 "
            "USD 1,284이 결제됐다고 합니다. 카드 끝자리는 7712예요. 막아주세요.")

res_haiku_v = W.run_verbose(graphs["Haiku"], victim_q, sc["script"], "Haiku(피해 신고 문구)")

In [ ]:
res_strands_v = W.run_verbose(graphs["StrandsDecider"], victim_q, sc["script"], "StrandsDecider(피해 신고 문구)")

### 표현을 바꿔가며 D1만 비교

문구 3종(피해 신고 2종 + 중립 1종)을 D1에만 넣어 양쪽 판정을 나란히 봅니다. label과 함께 **confidence가 문구의 애매함에 따라 달라지는지** 확인하세요 — 감정이 실린 신고 문구에서는 중립 문구보다 confidence가 낮게 나오는 것이 정상입니다. 이렇게 **보정된(calibrated) 신뢰도**라야 threshold 운용이 가능합니다.

In [ ]:
from IPython.display import Markdown, display
from agent.prompts import D1_QUESTION
from agent.state import D1_CHOICES
from agent.decisions import SystemOneDecision, HaikuDecision
from common import BEDROCK_MODEL_IDS, PRICING

sd = SystemOneDecision()
hk = HaikuDecision(client, BEDROCK_MODEL_IDS["haiku-4-5"], PRICING["haiku-4-5"])
probes = [
    victim_q,
    "오늘 새벽에 해외 결제 승인 알림이 왔는데 기억에 없는 거래예요. 확인해 주세요.",
    sc["question"],   # 본 시나리오의 중립 문구
]
rows = ["| 문구 | StrandsDecider D1 (confidence) | Haiku D1 |", "|---|---|---|"]
for t in probes:
    r_sd = sd.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    r_hk = hk.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    rows.append(f"| {t[:38]}... | {r_sd.label} ({r_sd.confidence:.3f}) | {r_hk.label} |")
display(Markdown("\n".join(rows)))

**관찰 포인트**
- 전용 decision model인 strands-decider E4B는 피해 신고 문구도 **in_scope로 정상 판정**합니다 — 범용 소형 모델을 zero-shot으로 썼을 때 흔한 과차단이 재현되지 않습니다. 판단 태스크로 적응(adapt)된 모델의 이점입니다.
- 더 중요한 것은 **confidence의 거동**입니다: 감정이 실린 신고 문구에서는 중립 문구보다 confidence가 낮아집니다. 보정된 신뢰도이므로 "confidence threshold 미달 D1만 상위 모델(LLM)로 escalate"하는 **cascade 설계**가 실제로 동작합니다.
- 운영 설계 원칙은 그대로입니다: **안전 판단(D1)은 임계값 기반 escalation을 두고**, routing·검토류(D2-D8)는 decision layer가 전담 — threshold가 작동하는 calibrated 모델이라 이 구성이 안전하게 성립합니다.
- 다음: `[KR]_05_bedrock_intelligent_routing.ipynb` — 판단이 아니라 "어느 생성 모델로 보낼지"를 고르는 관리형 라우터.
